# Picotron end-to-end demo: native pretraining, SFT, and DPO

This Kaggle notebook demonstrates two deliberately separate paths:

1. A **native Picotron ~1M decoder** pretrained on 100M FineWeb-Edu tokens, then sampled.
2. A real **HuggingFaceTB/SmolLM2-135M** model, SFT'd on Smol-SmolTalk and then DPO'd on UltraFeedback, with inference after each stage.

Run on a Kaggle GPU notebook. The 100M-token pretraining stage is a real run and can take substantial time.

## 1. Setup

Clone the public repository and install it in editable mode so the current `picotron`, `picotron_sft`, and `picotron_dpo` packages are used.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/kaggle/working/picotron')
REPO_URL = 'https://github.com/AndyMLAndAI/picotron.git'

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
os.chdir(REPO)

# Paste a gated-model / dataset token here only if needed. Public resources work with None.
HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running this notebook.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))
print('compute capability:', torch.cuda.get_device_capability(0))
if torch.cuda.get_device_capability(0)[0] != 7:
    print('Warning: this demo is tuned for a Turing T4 / fp16 path; verify precision and memory settings on this GPU.')


## 2. Native ~1M Picotron pretraining configuration

The tied GPT-2 embedding table dominates this tiny model: `50,257 × 20` parameters plus eight small decoder blocks produces roughly 1.06M parameters. The CLI automatically streams, tokenizes, and caches 100M FineWeb-Edu tokens before training.

In [ ]:
PRETRAIN_CONFIG = REPO / 'config_native_1m_fineweb.yaml'
PRETRAIN_CHECKPOINT = REPO / 'artifacts/native_1m_fineweb/model.safetensors'

# 12,208 × 32 × 256 = 100,007,936 token positions (approximately 100M tokens).
PRETRAIN_CONFIG.write_text('''
checkpoints:
  checkpoint_interval: 1000
  checkpoints_path: artifacts/native_1m_fineweb/model.safetensors
  save_final_state: true

model:
  # Turing/T4 policy: fp16, never bf16.
  dtype: float16
  compile_model: false
  triton_kernels:
    # These are the two Triton kernels with real training backward paths.
    rmsnorm: true
    swiglu: true
    # Keep the inference/fallback-only Triton paths disabled for training.
    rope: false
    attention: false
    cross_entropy: false
    adamw: false
  model_config:
    vocab_size: 50257
    hidden_size: 20
    intermediate_size: 80
    num_hidden_layers: 8
    num_attention_heads: 2
    num_key_value_heads: 1
    attention_type: gqa
    rope_theta: 1000000.0
    nope_layers: [3]
    sliding_window_size: 128
    moe_config: null
    kv_lora_rank: null
    tie_word_embeddings: true
    position_embedding_type: rope
    gradient_checkpointing: false

optimizer:
  learning_rate_scheduler:
    learning_rate: 0.0003
    lr_decay_style: constant
  optimizer_factory:
    name: adamw
    adam_beta1: 0.9
    adam_beta2: 0.95
    adam_eps: 1.0e-8
  weight_decay: 0.1
  clip_grad: 1.0

parallelism:
  dp: 1
  zero_stage: 0

tokens:
  sequence_length: 256
  micro_batch_size: 32
  train_steps: 12208

data:
  tokenizer_name: gpt2
  vocab_size: 50257
  token_cache_dir: data/token_cache
  datasets:
    - hf_name: HuggingFaceFW/fineweb-edu
      hf_config: sample-10BT
      target_tokens: 100000000
      text_field: text
      weight: 1.0
  num_workers: 4
  prefetch_factor: 2

logging:
  iteration_step_info_interval: 100
  file_logging: true
  file_logging_output_dir: logs

general:
  project: picotron
  run: native_1m_fineweb_100m
  seed: 1337
'''.lstrip(), encoding='utf-8')

print(PRETRAIN_CONFIG.read_text())


## 3. Pretrain on 100M FineWeb-Edu tokens

The first run materializes a deterministic uint16 token cache. Later runs reuse it. Notebook capture is automatically routed away from Rich Live, so the console does not fill with repeated tables.

In [ ]:
from picotron.config import load_config
from picotron.models.picotron_decoder import PicotronDecoderModel
from picotron.nn.feedforward import SwiGLU
from picotron.models.picotron_decoder import RMSNorm
from picotron.utils.hardware import detect_triton_support

# Fail before the long run rather than silently using a PyTorch fallback.
triton_report = detect_triton_support(enabled=True, device=DEVICE)
print('Triton preflight:', triton_report)
assert triton_report.installed, 'Triton is not installed; use a Kaggle PyTorch image that includes Triton.'
assert triton_report.hardware_compatible, 'This run requires a CUDA GPU with compute capability >= 7.0.'

# Execute both kernels and their backward paths once before committing to the 100M-token run.
probe_config = load_config(PRETRAIN_CONFIG)
assert probe_config.model.triton_kernels.rmsnorm and probe_config.model.triton_kernels.swiglu
probe_model = PicotronDecoderModel(probe_config).to(DEVICE).train()
probe_ids = torch.randint(0, probe_config.model.model_config.vocab_size, (1, 16), device=DEVICE)
with torch.autocast(device_type='cuda', dtype=torch.float16):
    probe_loss = probe_model(probe_ids).float().mean()
probe_loss.backward()
fallback_modules = [
    module.__class__.__name__
    for module in probe_model.modules()
    if isinstance(module, (RMSNorm, SwiGLU)) and module._triton_fallback_warned
]
assert not fallback_modules, f'Triton probe fell back to PyTorch in: {fallback_modules}'
print('Triton probe passed: RMSNorm + SwiGLU forward/backward executed without fallback.')
del probe_model, probe_loss
torch.cuda.empty_cache()

subprocess.run(
    ['picotron', '--config', str(PRETRAIN_CONFIG)],
    cwd=REPO,
    check=True,
)
assert PRETRAIN_CHECKPOINT.exists(), f'Missing final checkpoint: {PRETRAIN_CHECKPOINT}'
print('Native pretraining checkpoint:', PRETRAIN_CHECKPOINT)


## 4. Native-model inference after pretraining

The native generation helper intentionally recomputes the context at every token because Picotron’s native KV cache is not implemented yet. This is suitable for a short demo, not throughput benchmarking.

In [ ]:
from transformers import AutoTokenizer
from picotron.serialize import load_native_model

native_model = load_native_model(PRETRAIN_CHECKPOINT, device=DEVICE).eval()
native_tokenizer = AutoTokenizer.from_pretrained('gpt2', token=HF_TOKEN)
if native_tokenizer.pad_token_id is None:
    native_tokenizer.pad_token = native_tokenizer.eos_token

@torch.inference_mode()
def generate_native(prompt: str, max_new_tokens: int = 64) -> str:
    token_ids = native_tokenizer.encode(prompt, add_special_tokens=False)
    for _ in range(max_new_tokens):
        context = torch.tensor([token_ids[-256:]], dtype=torch.long, device=DEVICE)
        next_token = native_model(context)[0, -1].argmax().item()
        token_ids.append(next_token)
        if next_token == native_tokenizer.eos_token_id:
            break
    return native_tokenizer.decode(token_ids, skip_special_tokens=True)

for prompt in ('The future of machine learning is', 'Python functions are useful because'):
    print(f'\nPROMPT: {prompt}\n{generate_native(prompt)}')

del native_model
torch.cuda.empty_cache()


## 5. Load SmolLM2-135M and inspect baseline inference

SFT and DPO below use a real Hugging Face model, independently of the native 1M checkpoint. This demonstrates Picotron’s model-agnostic post-training adapters on a standard causal-LM logits interface.

In [ ]:
from dataclasses import replace
from itertools import islice

from datasets import Dataset, load_dataset
from transformers import AutoModelForCausalLM

from picotron.config import load_config
from picotron_sft import PicotronSFTConfig, PicotronSFTTrainer
from picotron_dpo import run_dpo

SMOLLM2_ID = 'HuggingFaceTB/SmolLM2-135M'
MAX_SEQ_LENGTH = 384
SFT_STEPS, SFT_BATCH_SIZE = 256, 4
DPO_STEPS, DPO_BATCH_SIZE = 256, 1

smollm_tokenizer = AutoTokenizer.from_pretrained(SMOLLM2_ID, token=HF_TOKEN, use_fast=True)
if smollm_tokenizer.pad_token_id is None:
    smollm_tokenizer.pad_token = smollm_tokenizer.eos_token
smollm_model = AutoModelForCausalLM.from_pretrained(
    SMOLLM2_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
).to(DEVICE)
smollm_model.config.pad_token_id = smollm_tokenizer.pad_token_id

def render_chat(messages, *, add_generation_prompt: bool) -> str:
    if getattr(smollm_tokenizer, 'chat_template', None):
        return smollm_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=add_generation_prompt
        )
    rendered = '\n'.join(f"{message['role']}: {message['content']}" for message in messages)
    return rendered + ('\nassistant:' if add_generation_prompt else '')

@torch.inference_mode()
def generate_hf(model, prompt: str, max_new_tokens: int = 96) -> str:
    encoded = smollm_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH)
    encoded = {name: value.to(DEVICE) for name, value in encoded.items()}
    generated = model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=smollm_tokenizer.pad_token_id,
        eos_token_id=smollm_tokenizer.eos_token_id,
    )
    new_tokens = generated[0, encoded['input_ids'].shape[1]:]
    return smollm_tokenizer.decode(new_tokens, skip_special_tokens=True)

DEMO_PROMPTS = [
    'Explain why the sky is blue in two sentences.',
    'Write a Python function that returns the nth Fibonacci number.',
]
for prompt in DEMO_PROMPTS:
    print(f'\nBASE PROMPT: {prompt}\n{generate_hf(smollm_model, prompt)}')


## 6. SFT SmolLM2 on Smol-SmolTalk

`HuggingFaceTB/smol-smoltalk` is the small-model SmolTalk subset and exposes a `messages` column. We stream only the 1,024 examples needed for this 256-step demonstration, format conversations with the model tokenizer, then hand the resulting `text` dataset to `PicotronSFTTrainer`.

In [ ]:
def take_streaming_dataset(name: str, split: str, count: int) -> Dataset:
    stream = load_dataset(name, split=split, streaming=True, token=HF_TOKEN)
    rows = list(islice(stream, count))
    assert len(rows) == count, f'Expected {count} rows from {name}/{split}, got {len(rows)}'
    return Dataset.from_list(rows)

raw_sft = take_streaming_dataset('HuggingFaceTB/smol-smoltalk', 'train', SFT_STEPS * SFT_BATCH_SIZE)
sft_dataset = raw_sft.map(
    lambda example: {'text': render_chat(example['messages'], add_generation_prompt=False)},
    remove_columns=raw_sft.column_names,
)
assert 'text' in sft_dataset.column_names and len(sft_dataset) == SFT_STEPS * SFT_BATCH_SIZE

# This config is only for Picotron's shared display; it does not alter the HF architecture.
native_display_config = load_config(PRETRAIN_CONFIG)
hf_config = smollm_model.config
hf_heads = int(hf_config.num_attention_heads)
hf_kv_heads = getattr(hf_config, 'num_key_value_heads', None)
hf_attention_type = 'gqa' if isinstance(hf_kv_heads, int) and hf_kv_heads < hf_heads else 'mha'
if hf_attention_type == 'mha':
    hf_kv_heads = None
hf_display_config = replace(
    native_display_config,
    model=replace(
        native_display_config.model,
        model_config=replace(
            native_display_config.model.model_config,
            vocab_size=len(smollm_tokenizer),
            hidden_size=int(hf_config.hidden_size),
            intermediate_size=int(hf_config.intermediate_size),
            num_hidden_layers=int(hf_config.num_hidden_layers),
            num_attention_heads=hf_heads,
            num_key_value_heads=hf_kv_heads,
            attention_type=hf_attention_type,
            nope_layers=(),
            sliding_window_size=None,
            moe_config=None,
            kv_lora_rank=None,
        ),
    ),
    tokens=replace(native_display_config.tokens, sequence_length=MAX_SEQ_LENGTH, micro_batch_size=SFT_BATCH_SIZE, train_steps=SFT_STEPS),
    data=replace(native_display_config.data, dataset_token_path=None, datasets=(), tokenizer_name=SMOLLM2_ID, vocab_size=len(smollm_tokenizer), num_workers=0),
    logging=replace(native_display_config.logging, iteration_step_info_interval=25, file_logging=True),
    general=replace(native_display_config.general, run='smollm2_135m_sft'),
)

smollm_model.config.use_cache = False
sft_trainer = PicotronSFTTrainer(
    model=smollm_model,
    tokenizer=smollm_tokenizer,
    train_dataset=sft_dataset,
    args=PicotronSFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=SFT_BATCH_SIZE,
        max_steps=SFT_STEPS,
        learning_rate=2e-5,
        logging_steps=25,
        max_seq_length=MAX_SEQ_LENGTH,
        device=DEVICE,
        display_config=hf_display_config,
    ),
)
sft_losses = sft_trainer.train()
assert len(sft_losses) == SFT_STEPS
SFT_DIR = REPO / 'artifacts/smollm2_135m_sft'
SFT_DIR.mkdir(parents=True, exist_ok=True)
smollm_model.config.use_cache = True
smollm_model.save_pretrained(SFT_DIR, safe_serialization=True)
smollm_tokenizer.save_pretrained(SFT_DIR)
print(f'SFT complete: first={sft_losses[0]:.4f}, last={sft_losses[-1]:.4f}; saved to {SFT_DIR}')


## 7. Inference after SFT

Use the same prompts to make the base → SFT comparison visible.

In [ ]:
for prompt in DEMO_PROMPTS:
    print(f'\nSFT PROMPT: {prompt}\n{generate_hf(smollm_model, prompt)}')


## 8. DPO SmolLM2 on UltraFeedback

UltraFeedback's `train_prefs` split stores `chosen` and `rejected` as message lists. We preserve the shared prompt history, take the final assistant message as each response, and let Picotron DPO tokenize/mask the response-only loss. The frozen reference is automatically copied from the post-SFT policy at DPO start.

In [ ]:
raw_dpo = take_streaming_dataset('HuggingFaceH4/ultrafeedback_binarized', 'train_prefs', 2 * DPO_STEPS * DPO_BATCH_SIZE)

def preference_row(example):
    chosen_messages = example['chosen']
    rejected_messages = example['rejected']
    return {
        'prompt': render_chat(chosen_messages[:-1], add_generation_prompt=True),
        'chosen': chosen_messages[-1]['content'],
        'rejected': rejected_messages[-1]['content'],
    }

dpo_dataset = raw_dpo.map(preference_row, remove_columns=raw_dpo.column_names)
dpo_dataset = dpo_dataset.filter(lambda row: bool(row['prompt'].strip()) and bool(row['chosen'].strip()) and bool(row['rejected'].strip()))
assert len(dpo_dataset) >= DPO_STEPS * DPO_BATCH_SIZE, 'Not enough valid UltraFeedback preference pairs for this run.'
dpo_dataset = dpo_dataset.select(range(DPO_STEPS * DPO_BATCH_SIZE))

dpo_display_config = replace(
    hf_display_config,
    tokens=replace(hf_display_config.tokens, micro_batch_size=DPO_BATCH_SIZE, train_steps=DPO_STEPS),
    general=replace(hf_display_config.general, run='smollm2_135m_dpo'),
)
smollm_model.config.use_cache = False
dpo_losses = run_dpo(
    model=smollm_model,
    dataset=dpo_dataset,
    tokenizer=smollm_tokenizer,
    beta=0.1,
    learning_rate=5e-6,
    batch_size=DPO_BATCH_SIZE,
    max_length=MAX_SEQ_LENGTH,
    num_steps=DPO_STEPS,
    device=DEVICE,
    display_config=dpo_display_config,
)
assert len(dpo_losses) == DPO_STEPS
DPO_DIR = REPO / 'artifacts/smollm2_135m_dpo'
DPO_DIR.mkdir(parents=True, exist_ok=True)
smollm_model.config.use_cache = True
smollm_model.save_pretrained(DPO_DIR, safe_serialization=True)
smollm_tokenizer.save_pretrained(DPO_DIR)
print(f'DPO complete: first={dpo_losses[0]:.4f}, last={dpo_losses[-1]:.4f}; saved to {DPO_DIR}')


## 9. Inference after DPO

Compare the same prompts one last time. A short demo run is evidence that the path executes; it is not a claim of benchmark-quality alignment.

In [ ]:
for prompt in DEMO_PROMPTS:
    print(f'\nDPO PROMPT: {prompt}\n{generate_hf(smollm_model, prompt)}')

print('\nArtifacts:')
for path in (PRETRAIN_CHECKPOINT, SFT_DIR, DPO_DIR):
    print(' -', path)
